## Cálculo de similitudes y asignación de claves de la Muestra 3

Una vez concluido el preprocesamiento de la **Muestra 3**, se desarrolló un algoritmo de coincidencia multivariable para identificar las correspondencias entre los vehículos de la muestra y los registros del catálogo vehicular. 

Para lograr un *match* óptimo, la metodología integró dos enfoques complementarios: el **coeficiente de Gower** (para validar características técnicas estructurales como el año, los ocupantes y la transmisión) y **medidas de similitud textual** (para evaluar la coincidencia descriptiva). Ambas métricas se integraron en una similitud global ponderada.

### 1. Preparación de variables de referencia

Para posibilitar la comparación, se preparó una copia del catálogo vehicular aplicando estandarizaciones clave previas a la ejecución del algoritmo:
* Se normalizaron los nombres de las columnas para evitar errores por caracteres especiales o acentos.
* Se estandarizó la variable `Ocupantes` en ambas bases, extrayendo únicamente los valores numéricos enteros (ej. "5 OCUP." se transformó a `5`).
* Se construyó una `descripcion_generada` para cada registro del catálogo concatenando las variables `MARCA`, `TIPO`, `VERSION` y `TRANSMISIÓN` (transformando los códigos `A` y `S` a `AUT` y `STD`, respectivamente). 
* La descripción original de la Muestra 3 se mantuvo intacta, conservando los metadatos integrados en el texto para su evaluación.

### 2. Similitud de Gower (Características estructurales)

El coeficiente de Gower se utilizó exclusivamente para comparar las variables técnicas duras de los vehículos. Para la Muestra 3, se definieron **tres variables como categóricas**, exigiendo coincidencia exacta: `Modelo` (año), `Transmisión` y `Ocupantes`.

Para facilitar el cálculo algorítmico, las categorías se codificaron conjuntamente, identificando los valores nulos con el código `-1`. La similitud se obtiene promediando las coincidencias observadas, excluyendo del cálculo matemático las variables con información faltante:

$$S_G(i,j)=\frac{\sum_{k=1}^{p}\delta_{ijk}s_{ijk}}{\sum_{k=1}^{p}\delta_{ijk}}$$

Donde:
* $S_G(i,j)$ es la similitud de Gower entre los vehículos $i$ y $j$.
* $p=3$ es el número de variables categóricas.
* $s_{ijk}=1$ si coinciden exactamente en la variable $k$, y $0$ en caso contrario.
* $\delta_{ijk}=1$ si ambos tienen datos disponibles para la variable $k$, y $0$ si falta información.

El resultado oscila entre 0 y 1, aportando rigor técnico a la evaluación.

### 3. Similitud textual (Descripción)

Para complementar las características técnicas, se evaluó la redacción de las descripciones utilizando la biblioteca `RapidFuzz` mediante dos enfoques:

1. **Preselección (Token Set Ratio):** Identifica términos compartidos sin importar el orden, permitiendo preseleccionar descripciones altamente compatibles de forma rápida.
2. **Evaluación final (Combinada):** Para la asignación definitiva, se integró el *Token Sort Ratio* (que ordena alfabéticamente las palabras y penaliza diferencias estructurales). La métrica textual definitiva equilibró ambos comportamientos:

$$S_T(i,j)=0.75S_{\text{set}}(i,j)+0.25S_{\text{sort}}(i,j)$$

Se otorgó un **75 % al enfoque de palabras compartidas (Set) y un 25 % al enfoque estricto (Sort)**, reduciendo así puntuaciones artificialmente perfectas por exceso de texto. 

### 4. Construcción de la similitud global ponderada

Para elegir al mejor candidato, la decisión final integró el rigor técnico y la coincidencia descriptiva asignando un peso equitativo a cada métrica:

| Componente | Ponderación |
|---|---:|
| Similitud de Gower (Año, Transmisión, Ocupantes) | 50 % |
| Similitud de Descripción (Texto completo) | 50 % |
| **Total** | **100 %** |

La similitud global se calculó mediante la media ponderada estricta de estos dos componentes:

$$S_{\text{global}}(i,j)=0.50S_G(i,j)+0.50S_D(i,j)$$

### 5. Algoritmo de selección y asignación de claves (Claveteo)

Para optimizar el rendimiento, el proceso de asignación de identificadores (claveteo) se ejecutó sobre cada vehículo de la muestra mediante el siguiente embudo metodológico:

1. **Filtro excluyente (Ocupantes):** Como requerimiento primario, se descartaron del catálogo los registros que no coincidían con la variable `Ocupantes` del vehículo analizado.
2. **Preselección (Top 15):** Utilizando el *Token Set Ratio*, se calcularon las similitudes textuales de las descripciones del catálogo filtrado, aislando las 15 descripciones distintas con mayor puntaje.
3. **Cálculo Global:** A los candidatos de este Top 15 se les aplicó la función de *Gower* y la *Similitud textual combinada*, unificándolas en la $S_{\text{global}}$ (50/50).
4. **Eliminación de duplicidades:** Si varios registros del catálogo compartían la misma descripción, se conservó únicamente aquel con el mayor puntaje unificado.
5. **Asignación:** Se seleccionó el candidato con mayor puntaje global. Si este superaba el **umbral estricto de 0.80 (80 %)**, se extrajo su identificador (`ID`) para asignarlo en la columna `ID_asignado`. En caso contrario, el registro quedó sin asignar.

### 6. Resultados y alcance

El proceso concluyó con la generación del archivo de salida `Muestra_3_claveteada.xlsx`. 

Este modelo híbrido demostró ser altamente efectivo al exigir que un *match* válido no solo tuviera una descripción similar (texto), sino que estructuralmente coincidiera en su año, capacidad de pasajeros y tipo de transmisión (Gower). Las claves asignadas superaron con éxito el umbral del 80 %, dejando los casos atípicos aislados para auditoría manual.